### Download ckpt

In [4]:
# Azure Machine Learning workspace details:
subscription = 'f920ee3b-6bdc-48c6-a487-9e0397b69322'
resource_group = 'msan-aml'
workspace = 'msan-retrieval-ranking-aml'
datastore_name = 'bingads_algo_prod_networkprotection_c08'
path_on_datastore = '/shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1RankingV3/Checkpoint_ST/checkpoint-pinSage/event_doc_text-PinSage/'

# long-form Datastore uri format:
uri = f'azureml://subscriptions/{subscription}/resourcegroups/{resource_group}/workspaces/{workspace}/datastores/{datastore_name}/paths/{path_on_datastore}'

In [5]:
from azureml.fsspec import AzureMachineLearningFileSystem

# instantiate file system using following URI
fs = AzureMachineLearningFileSystem(uri)

fs.ls() # list folders/files in datastore 'datastorename'

['shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1RankingV3/Checkpoint_ST/checkpoint-pinSage/event_doc_text-PinSage/config.json',
 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1RankingV3/Checkpoint_ST/checkpoint-pinSage/event_doc_text-PinSage/event_doc_text',
 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1RankingV3/Checkpoint_ST/checkpoint-pinSage/event_doc_text-PinSage/event_doc_text_model.bin',
 'shares/Bingads.algo.prod.IntentMatching/Data/users/shenqian/MSAN/L1RankingV3/Checkpoint_ST/checkpoint-pinSage/event_doc_text-PinSage/vocab.txt']

### Load Model

In [ ]:
from model.PinSageEncoder import PinSageEncoder

model = PinSageEncoder.load("./ckpt")

/anaconda/envs/hstu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Inference with model

In [ ]:
import torch

max_length = 32 # need to verify this, I feel 128 is a better choice which is aligned with model's comment
test_text = "Design Art River By The Canyon Landscape I - Canyon Canvas Wall Art 20.0 H x 12.0 W x 1.0 D in Canvas, in Silver Single Picture Framed | Wayfair [SEP] www.wayfair.com  Design Art  River By The Canyon Landscape I Canyon Canvas Wall Art CBUW4442 L1318 K~CBUW4442.html"

tokenizer = model.tokenizer
tokens = tokenizer(
    test_text,
    padding="max_length",
    max_length=max_length,
    truncation=True,
    return_tensors="pt",
    add_special_tokens=False
)
print(tokens)

input_ids = tokens["input_ids"]
attention_mask = tokens["attention_mask"]

# model.to(device)
model.eval()

with torch.no_grad():
    embeddings = model(input_ids, attention_mask)
    print(embeddings.shape)
    print(embeddings)

{'input_ids': tensor([[12617, 10901, 10954, 10151, 10103, 19975, 34029,   151,   118, 19975,
         83406, 15813, 10901, 10200,   119,   121,   150,   166, 10189,   119,
           121,   165,   166,   122,   119,   121,   146, 10104, 83406,   117,
         10104, 15143]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 64])
tensor([[ 0.5309,  0.1074,  0.3974, -0.3747, -0.1219, -0.3042,  0.3987,  0.2582,
         -0.2131,  0.2764,  0.3359,  0.0235,  0.1375,  0.1644, -0.1406, -0.1322,
          0.1382, -0.1164,  0.0770, -0.0625,  0.0862, -0.0466,  0.0756,  0.0248,
         -0.0841,  0.4064, -0.0946, -0.1322,  0.2228, -0.0448,  0.3300, -0.0360,
         -0.0402,  0.1987,  0.1700,  0.1728,  0.0618, -0.0753,  0.4552, -0.4012,
          0.2585,  0.0038,  